# RAG System - Query & Retrieval Demo

This notebook demonstrates the query and retrieval stage of the RAG pipeline:
1. **Load Vector Store** - Load the pre-built FAISS index
2. **Test Retrieval** - Query the vector store and inspect results
3. **LLM Response** - Generate answers with citations (TODO)
4. **Evaluation** - Metrics on test questions (TODO)

Prerequisites: Run `01_indexing_demo.ipynb` first to create the vector store.

---
## Setup

In [1]:
# Imports
import os
import numpy as np
from pathlib import Path
import json

from rag_system.vector_store import FAISSVectorStore, VectorStoreConfig, SearchResult
from rag_system.embeddings import Embedder, EmbeddingConfig
from rag_system.models import DocumentChunk

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd()
INDEX_DIR = PROJECT_ROOT / "faiss_index"

print(f"Project root:     {PROJECT_ROOT}")
print(f"Index directory:  {INDEX_DIR}")
print(f"Index exists:     {INDEX_DIR.exists()}")

Project root:     c:\Development\git\rag_achiles
Index directory:  c:\Development\git\rag_achiles\faiss_index
Index exists:     True


---
## 1. Load Vector Store

Load the pre-built FAISS index and metadata from disk.

In [3]:
# Load vector store (auto-loads existing index if found)
print("Loading vector store...\n")

threshold = 0.6 # set similiratity threshold
vs_config = VectorStoreConfig(similarity_threshold=threshold)
vector_store = FAISSVectorStore(config=vs_config)

print(f"✓ Vector store loaded successfully!")
print(f"\nStatistics:")
stats = vector_store.get_stats()
for key, value in stats.items():
    print(f"  {key:20s}: {value}")

Loading vector store...

Loaded index with 441 vectors from faiss_index\vector_index.faiss
✓ Vector store loaded successfully!

Statistics:
  total_vectors       : 441
  dimension           : 384
  index_type          : FAISS-IndexFlatIP
  total_chunks        : 441
  similarity_threshold: 0.6
  index_path          : faiss_index\vector_index.faiss
  metadata_path       : faiss_index\metadata.pkl


### Initialize Embedder

Load the same embedding model used during indexing to embed queries.

In [4]:
# Load embedder (same model as indexing)
print("Loading embedder...\n")

embedder = Embedder()

print(f"\nEmbedder ready:")
print(f"  Model:     {embedder.config.model_name}")
print(f"  Dimension: {embedder.dimension}")

Loading embedder...

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2...


c:\Development\environments\aida-venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✓ Model loaded (dimension: 384)

Embedder ready:
  Model:     paraphrase-multilingual-MiniLM-L12-v2
  Dimension: 384


---
## 2. Test Retrieval (No LLM)

Let's test the vector store retrieval with sample queries to verify semantic search works correctly.

### Helper Functions

Create reusable functions for querying and analyzing results.

In [5]:
def query_and_display(query: str, top_k: int = 5, show_full_text: bool = False, show_rerank: bool = True):
    """
    Query vector store and display results.
    
    Args:
        query: Search query
        top_k: Number of results to retrieve
        show_full_text: If True, show complete chunk text; if False, show preview
        show_rerank: If True, show rerank score when available
    """
    query_emb = embedder.embed_query(query)
    results = vector_store.search(query_emb, top_k=top_k)
    
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)
    print(f"Found {len(results)} results (top-{top_k})\n")
    
    if not results:
        print("No results found above similarity threshold.\n")
        return results
    
    for i, result in enumerate(results, 1):
        chunk = result.chunk
        score_str = f"Score: {result.score:.4f}"
        
        # Show rerank score if available and requested
        if show_rerank and result.rerank_score is not None:
            score_str += f" | Rerank: {result.rerank_score:.3f}"
        
        print(f"[{i}] {score_str} | {chunk.document} | Page {chunk.page}")
        
        if show_full_text:
            print(f"    {chunk.text}\n")
        else:
            print(f"    {chunk.text[:150]}...\n")
    
    return results


def analyze_scores(queries: list, top_k: int = 5):
    """
    Analyze similarity scores across multiple queries.
    
    Args:
        queries: List of query strings
        top_k: Number of results per query
    """
    print("Similarity Score Analysis")
    print("=" * 80)
    
    all_scores = []
    for query in queries:
        query_emb = embedder.embed_query(query)
        results = vector_store.search(query_emb, top_k=top_k)
        
        if results:
            scores = [r.score for r in results]
            all_scores.extend(scores)
            print(f"\n{query}")
            print(f"  Top: {max(scores):.3f} | Low: {min(scores):.3f} | Avg: {sum(scores)/len(scores):.3f} | Results: {len(results)}")
    
    if all_scores:
        print("\n" + "=" * 80)
        print(f"Overall: Max={max(all_scores):.3f}, Min={min(all_scores):.3f}, Avg={sum(all_scores)/len(all_scores):.3f}, Std={np.std(all_scores):.3f}")
        print("=" * 80)

### Sample Queries

Test various queries on the corpus.

In [6]:
# Test various queries based on actual document content
query_and_display("¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?", top_k=3)
query_and_display("¿Cuáles son los cuatro ejes del marco estratégico?", top_k=3)
query_and_display("¿Qué organizaciones participaron en el proceso de consulta pública?", top_k=3)
query_and_display("¿Cómo se evalúa la estrategia de accesibilidad universal?", top_k=3)

QUERY: ¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?
Found 3 results (top-3)

[1] Score: 0.7387 | resumengrupomotor260224.pdf | Page 1
    Para concretar las acciones que podrían incluirse en el nuevo plan es preciso tener en
cuenta y casar los siguientes elementos:
- Los objetivos de la ...

[2] Score: 0.7292 | Resumen_9_reunion_marzo2026.pdf | Page 1
    La directora general (DG) de Transparencia y Calidad anuncia la disponibilidad del informe
intermedio de seguimiento del IV Plan, que es u na de las o...

[3] Score: 0.7148 | ResumenReunionGA_20231124.pdf | Page 6
    Los objetivos fueron
 Reforzar a transparencia y la rendición de cuentas
 Consolidar un modelo de participación inclusivo, moderno y eficaz
 Recupe...

QUERY: ¿Cuáles son los cuatro ejes del marco estratégico?
Found 3 results (top-3)

[1] Score: 0.6823 | Resumen_grupo_motor_041024.pdf | Page 1
     En relación con el marco estratégico:
o Se acuerda explicar que el marco estratégico se define precisame

[SearchResult(chunk=DocumentChunk(text='La accesibilidad universal es una característica que afecta e implica a todas las unidades organizativas del\nayuntamiento. Para su impulso y desarrollo, el 10 de noviembre de 2022 , se aprueba por acuerdo de Junta\nde Gobierno, el Plan Estratégico de Accesibilidad Universal para la ciudad de Madrid (PEAUM ), como hoja\nde ruta a seguir en los próximos años, que busca una mejora en la gestión con un enfoque transversal y\nuniversal .\nPara el desarrollo del PEAUM se aplicó una metodología participativa de percepción de la accesibilidad\nuniversal con una doble mirada, a nivel de organización (nivel interno) y a nivel externo (nivel de ciudadanía).\nLa fase de análisis de la situación y su diagnóstico arrojó unos resultados que permitieron definir las cinco', document='resumen_grupo_motor110424.pdf', page=8, chunk_index=31), score=0.8134716749191284, rerank_score=None),
 SearchResult(chunk=DocumentChunk(text='Sobre la propuesta de evaluación de la

### Score Analysis Across Queries

In [7]:
# Analyze similarity scores across different query types
test_queries = [
    "¿Qué compromisos incluye el cuarto plan?",
    "¿Cuáles son los ejes de transparencia y datos abiertos?",
    "¿Qué es la Escuela de Gobierno Abierto?",
    "¿Quiénes asistieron a las reuniones del grupo motor?",
    "¿Cómo se realiza el seguimiento del plan?",
    "¿Qué papel tiene la participación ciudadana en el plan?",
]

analyze_scores(test_queries, top_k=5)

Similarity Score Analysis

¿Qué compromisos incluye el cuarto plan?
  Top: 0.730 | Low: 0.670 | Avg: 0.695 | Results: 5

¿Cuáles son los ejes de transparencia y datos abiertos?
  Top: 0.784 | Low: 0.606 | Avg: 0.670 | Results: 3

¿Qué es la Escuela de Gobierno Abierto?
  Top: 0.659 | Low: 0.659 | Avg: 0.659 | Results: 1

¿Cómo se realiza el seguimiento del plan?
  Top: 0.699 | Low: 0.678 | Avg: 0.688 | Results: 3

¿Qué papel tiene la participación ciudadana en el plan?
  Top: 0.744 | Low: 0.701 | Avg: 0.725 | Results: 5

Overall: Max=0.784, Min=0.606, Avg=0.696, Std=0.043


---
## 3. Re-ranking with Cross-Encoder

Test the re-ranker to see how it improves results over bi-encoder alone.

### Initialize Re-ranker

In [8]:
from rag_system.reranker import Reranker, RerankerConfig

# Load cross-encoder re-ranker
print("Loading re-ranker...\n")
reranker = Reranker()

print("✓ Re-ranker ready")
print(f"  Model: {reranker.config.model_name}")
print(f"  Top-N: {reranker.config.top_n}")

Loading re-ranker...

Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-12-v2...
✓ Cross-encoder loaded
✓ Re-ranker ready
  Model: cross-encoder/ms-marco-MiniLM-L-12-v2
  Top-N: 5


### Compare Bi-encoder vs Cross-encoder Rankings

In [9]:
# Test query with re-ranking
query = "¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?"

# Step 1: Get candidates with bi-encoder (top-20)
query_emb = embedder.embed_query(query)
candidates = vector_store.search(query_emb, top_k=20)

print(f"Query: {query}")
print("=" * 80)
print(f"\nBi-encoder retrieved {len(candidates)} candidates\n")

# Step 2: Re-rank with cross-encoder
reranked = reranker.rerank(query, candidates, top_n=5)

print("Top-5 BEFORE re-ranking (bi-encoder only):")
print("-" * 80)
for i, result in enumerate(candidates[:5], 1):
    print(f"{i}. Score: {result.score:.4f} | {result.chunk.document[:45]}... | Page {result.chunk.page}")

print("\n" + "=" * 80)
print("Top-5 AFTER re-ranking (cross-encoder):")
print("-" * 80)
for i, result in enumerate(reranked, 1):
    print(f"{i}. Bi-encoder: {result.retrieval_score:.4f} | Cross-encoder: {result.rerank_score:.3f}")
    print(f"   {result.chunk.document[:45]}... | Page {result.chunk.page}")

print("\n" + "=" * 80)

Query: ¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?

Bi-encoder retrieved 20 candidates

Top-5 BEFORE re-ranking (bi-encoder only):
--------------------------------------------------------------------------------
1. Score: 0.7387 | resumengrupomotor260224.pdf... | Page 1
2. Score: 0.7292 | Resumen_9_reunion_marzo2026.pdf... | Page 1
3. Score: 0.7148 | ResumenReunionGA_20231124.pdf... | Page 6
4. Score: 0.7102 | Resumen_grupo_motor_041024.pdf... | Page 1
5. Score: 0.7029 | resumen_grupo_motor110424.pdf... | Page 1

Top-5 AFTER re-ranking (cross-encoder):
--------------------------------------------------------------------------------
1. Bi-encoder: 0.7029 | Cross-encoder: 7.362
   resumen_grupo_motor110424.pdf... | Page 1
2. Bi-encoder: 0.6736 | Cross-encoder: 7.279
   Resumen_grupo_motor_14032025.pdf... | Page 4
3. Bi-encoder: 0.7102 | Cross-encoder: 7.055
   Resumen_grupo_motor_041024.pdf... | Page 1
4. Bi-encoder: 0.6685 | Cross-encoder: 6.180
   resumen_grupo_motor11

### Analyze Rank Changes

In [10]:
# Compare rankings and see what changed
comparison = reranker.compare_rankings(query, candidates)

print("Re-ranking Impact Analysis")
print("=" * 80)
print(f"Total candidates:  {comparison['total_candidates']}")
print(f"Top-5 overlap:     {comparison['top_5_overlap']}/5 ({comparison['overlap_percentage']:.0f}%)")

print("\nBiggest rank changes (top-10):")
print("-" * 80)
for change in comparison['rank_changes'][:10]:
    direction = "↑" if change['change'] > 0 else "↓"
    abs_change = abs(change['change'])
    print(f"{direction} Rank {change['old_rank']:2d} → {change['new_rank']:2d} ({abs_change:+2d}): {change['document'][:50]}... | Page {change['page']}")
    print(f"   Bi: {change['retrieval_score']:.3f} | Cross: {change['rerank_score']:.3f}")
    
print("=" * 80)

Re-ranking Impact Analysis
Total candidates:  20
Top-5 overlap:     1/5 (20%)

Biggest rank changes (top-10):
--------------------------------------------------------------------------------
↑ Rank 18 →  4 (+14): resumen_grupo_motor110424.pdf... | Page 1
   Bi: 0.668 | Cross: 6.180
↓ Rank  5 → 17 (+12): Resumen_9_reunion_marzo2026.pdf... | Page 1
   Bi: 0.729 | Cross: 0.346
↓ Rank  6 → 18 (+12): Resumen_9_reunion_marzo2026.pdf... | Page 1
   Bi: 0.701 | Cross: -0.897
↓ Rank  6 → 16 (+10): resumengrupomotor260224.pdf... | Page 1
   Bi: 0.739 | Cross: 2.360
↑ Rank 11 →  2 (+9): Resumen_grupo_motor_14032025.pdf... | Page 4
   Bi: 0.674 | Cross: 7.279
↑ Rank 20 → 12 (+8): Resumen_grupo_motor_14032025.pdf... | Page 5
   Bi: 0.657 | Cross: 4.802
↓ Rank 13 → 19 (+6): resumen_grupo_motor110424.pdf... | Page 2
   Bi: 0.671 | Cross: -1.102
↑ Rank 10 →  5 (+5): resumen_grupo_motor110424.pdf... | Page 24
   Bi: 0.685 | Cross: 5.862
↑ Rank 16 → 11 (+5): resumen_grupo_motor110424.pdf... | Page 23
  

### Inspect Re-ranked Results Content

In [11]:
# Show the actual text of top re-ranked results
print("Top-3 re-ranked results (full text):")
print("=" * 80)

for i, result in enumerate(reranked[:3], 1):
    print(f"\n[{i}] Cross-encoder score: {result.rerank_score:.3f}")
    print(f"    {result.chunk.document} | Page {result.chunk.page}")
    print(f"    {result.chunk.text[:300]}...")
    print("-" * 80)

Top-3 re-ranked results (full text):

[1] Cross-encoder score: 7.362
    resumen_grupo_motor110424.pdf | Page 1
    El día 11 de abril de 2024 tiene lugar una nueva reunión del grupo motor para el diseño del
cuarto plan de gobierno abierto del Ayuntamiento de Madrid .
Asisten a la reunión las personas que se indican en el anexo 1 a este documento .
El objetivo de la reunión es la puesta en común de posibles compr...
--------------------------------------------------------------------------------

[2] Cross-encoder score: 7.279
    Resumen_grupo_motor_14032025.pdf | Page 4
    - Compromiso de evaluación de la estrategia de accesibilidad universal. Por parte de
la directora de la general de accesibilidad se explica que los principales hitos de este
compromiso se iniciarán más adelante de acuerdo con el propio cronograma
incluido en el cuarto plan de gobierno abierto . El p...
--------------------------------------------------------------------------------

[3] Cross-encoder score: 7.055


### Inspect Full Text

In [12]:
# View complete text of retrieved chunks for detailed inspection
query_and_display("¿Qué es el compromiso de comunicación clara?", top_k=2, show_full_text=True)

QUERY: ¿Qué es el compromiso de comunicación clara?
Found 2 results (top-2)

[1] Score: 0.7669 | Resumen_grupo-motor130924.pdf | Page 2
    Las aportaciones pueden resumirse del siguiente modo:
• la importancia y relevancia de este compromiso de comunicación clara : usar un
lenguaje sencillo que cualquiera pueda entender y dejar los tecnicismos, explicando
los términos o conceptos legales que sean necesarios para tomar decisiones
informadas y participar activamente en la sociedad y fomentar la transparencia .
• La necesid ad de que exista mayor transparencia (derecho a conocer): información
veraz, completa y a disposición de toda la ciudadanía.
• Algunos comentarios inciden en la importancia del contenido, además de la
comunicación clara: “La claridad es importante, pero el contenido lo es más” .
• También algunos se refieren a que la mejora de la claridad de la comunicación

[2] Score: 0.6920 | Resumen_8_reunion_grupo_motor_noviembre2025.pdf | Page 5
    y la Adolescencia) . Por el mo

[SearchResult(chunk=DocumentChunk(text='Las aportaciones pueden resumirse del siguiente modo:\n• la importancia y relevancia de este compromiso de comunicación clara : usar un\nlenguaje sencillo que cualquiera pueda entender y dejar los tecnicismos, explicando\nlos términos o conceptos legales que sean necesarios para tomar decisiones\ninformadas y participar activamente en la sociedad y fomentar la transparencia .\n• La necesid ad de que exista mayor transparencia (derecho a conocer): información\nveraz, completa y a disposición de toda la ciudadanía.\n• Algunos comentarios inciden en la importancia del contenido, además de la\ncomunicación clara: “La claridad es importante, pero el contenido lo es más” .\n• También algunos se refieren a que la mejora de la claridad de la comunicación', document='Resumen_grupo-motor130924.pdf', page=2, chunk_index=6), score=0.7669389843940735, rerank_score=None),
 SearchResult(chunk=DocumentChunk(text='y la Adolescencia) . Por el momento, s e destaca 

### Edge Case: Off-Topic Queries

In [13]:
# Test off-topic queries (should return no/low results)
print("Testing Off-Topic Queries (should have low/no results)")
print("=" * 80)

off_topic = [
    "¿Cuál es la receta de la paella valenciana?",
    "¿Qué tiempo hace en Barcelona en verano?",
]

for q in off_topic:
    results = query_and_display(q, top_k=2)
    if results and results[0].score > 0.75:
        print("⚠ WARNING: Unexpectedly high score for off-topic query!\n")
    elif not results:
        print("✓ Correctly filtered: No results above threshold\n")
    else:
        print("✓ Low scores as expected\n")

Testing Off-Topic Queries (should have low/no results)


QUERY: ¿Cuál es la receta de la paella valenciana?
Found 0 results (top-2)

No results found above similarity threshold.

✓ Correctly filtered: No results above threshold

QUERY: ¿Qué tiempo hace en Barcelona en verano?
Found 0 results (top-2)

No results found above similarity threshold.

✓ Correctly filtered: No results above threshold

